In [0]:
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 60.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

os.environ["KAGGLE_USERNAME"] = ""
os.environ["KAGGLE_KEY"] = ""

print("Kaggle credentials configured!")

Kaggle credentials configured!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [00:35<00:00, 129MB/s]


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-36db71cb-7fb4-4633-82ac-56 nogroup 8.4G Jan 16 13:45 2019-Nov.csv
-rwxrwxrwx 1 spark-36db71cb-7fb4-4633-82ac-56 nogroup 5.3G Jan 16 13:48 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 16 13:34 delta
-rwxrwxrwx 1 spark-36db71cb-7fb4-4633-82ac-56 nogroup 4.3G Jan 16 13:45 ecommerce-behavior-data-from-multi-category-store.zip
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 16 13:34 outputs


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 spark-36db71cb-7fb4-4633-82ac-56 nogroup 8.4G Jan 16 13:45 2019-Nov.csv
-rwxrwxrwx 1 spark-36db71cb-7fb4-4633-82ac-56 nogroup 5.3G Jan 16 13:48 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 16 13:34 delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 16 13:34 outputs


In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
source_csv = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"
base_vol = "/Volumes/workspace/ecommerce/ecommerce_data"
bronze_path = f"{base_vol}/delta/bronze/events"
silver_path = f"{base_vol}/delta/silver/events"
gold_path   = f"{base_vol}/delta/gold/product_perf"
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_gold")

bronze_table = "workspace.ecommerce_bronze.events"
silver_table = "workspace.ecommerce_silver.events"
gold_table   = "workspace.ecommerce_gold.product_perf"

In [0]:
bronze = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(source_csv)
          .withColumn("ingestion_ts", F.current_timestamp())
          .withColumn("source_file", F.lit(source_csv)))
(bronze.write
 .format("delta")
 .mode("overwrite")
 .save(bronze_path))
(bronze.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(bronze_table))

display(spark.table(bronze_table).limit(5))

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,ingestion_ts,source_file
2019-10-17T17:08:43.000Z,view,16800164,2053013558316237377,furniture.kitchen.table,null,156.25,517526003,0fb659fd-a4b3-426b-a9a0-2e1bbbd1bbc8,2026-01-16T13:50:14.275Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv
2019-10-17T17:08:43.000Z,view,17500208,2053013558752445019,null,missha,20.34,556784046,8a09a0af-9304-44d8-a643-4469c5b9b93f,2026-01-16T13:50:14.275Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv
2019-10-17T17:08:43.000Z,view,26200162,2053013563693335403,null,null,178.64,513871247,25714cd1-f1e9-4d52-9989-12cb4434e1af,2026-01-16T13:50:14.275Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv
2019-10-17T17:08:43.000Z,view,12705692,2053013553559896355,null,triangle,73.36,535126914,81f3adeb-9534-483a-adc3-6cd644688d1b,2026-01-16T13:50:14.275Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv
2019-10-17T17:08:43.000Z,view,12300394,2053013556311359947,construction.tools.drill,null,46.82,530203233,d9d2f30d-102c-4090-93a3-d94ffc9ef350,2026-01-16T13:50:14.275Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv


In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

silver = (bronze_df
          .withColumn("event_ts", F.to_timestamp("event_time"))
          .withColumn("event_date", F.to_date("event_ts"))
          .withColumn("price", F.col("price").cast("double"))
          .filter(F.col("event_ts").isNotNull())
          .filter(F.col("event_type").isNotNull())
          .filter(F.col("user_session").isNotNull())
          .filter((F.col("price").isNull()) | ((F.col("price") > 0) & (F.col("price") < 10000)))
          .dropDuplicates(["user_session", "event_time", "event_type", "product_id"])
          .withColumn(
              "price_tier",
              F.when(F.col("price").isNull(), F.lit("unknown"))
               .when(F.col("price") < 10, F.lit("budget"))
               .when(F.col("price") < 50, F.lit("mid"))
               .otherwise(F.lit("premium"))
          )
         )

(silver.write.format("delta").mode("overwrite").save(silver_path))
(silver.write.format("delta").mode("overwrite").saveAsTable(silver_table))

display(spark.table(silver_table).limit(5))

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,ingestion_ts,source_file,event_ts,event_date,price_tier
2019-10-13T06:28:07.000Z,view,29900054,2059484601444729123,null,peda,2557.59,513219899,9789da6d-4f86-463a-adf7-8ac3db11def3,2026-01-16T13:49:40.593Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv,2019-10-13T06:28:07.000Z,2019-10-13,premium
2019-10-13T06:30:19.000Z,view,1005099,2053013555631882655,electronics.smartphone,samsung,145.15,548709727,b5d8cb9e-97c9-45dd-9178-615679d1ca4c,2026-01-16T13:49:40.593Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv,2019-10-13T06:30:19.000Z,2019-10-13,premium
2019-10-13T06:30:28.000Z,view,15900124,2053013558190408249,null,tefal,23.14,558702304,275123ad-729b-4029-baa6-1e535a399760,2026-01-16T13:49:40.593Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv,2019-10-13T06:30:28.000Z,2019-10-13,mid
2019-10-13T06:31:13.000Z,view,17301264,2053013553853497655,null,desigual,54.99,512467152,2f661dd5-0753-4636-92d7-8b93e37de4cd,2026-01-16T13:49:40.593Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv,2019-10-13T06:31:13.000Z,2019-10-13,premium
2019-10-13T06:31:20.000Z,view,1005143,2053013555631882655,electronics.smartphone,apple,1565.03,516048756,2625a4ea-9b23-4b1a-b318-ce3cf3ef12f5,2026-01-16T13:49:40.593Z,/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv,2019-10-13T06:31:20.000Z,2019-10-13,premium


In [0]:
silver_df = spark.read.format("delta").load(silver_path)

gold = (silver_df.groupBy("product_id")
        .agg(
            F.countDistinct(F.when(F.col("event_type") == "view", F.col("user_id"))).alias("unique_viewers"),
            F.countDistinct(F.when(F.col("event_type") == "purchase", F.col("user_id"))).alias("unique_purchasers"),
            F.round(F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0))), 2).alias("revenue")
        )
        .withColumn(
            "conversion_rate_pct",
            F.when(F.col("unique_viewers") == 0, F.lit(0.0))
             .otherwise(F.round((F.col("unique_purchasers") / F.col("unique_viewers")) * 100, 4))
        )
       )

(gold.write.format("delta").mode("overwrite").save(gold_path))
(gold.write.format("delta").mode("overwrite").saveAsTable(gold_table))

display(spark.table(gold_table).orderBy(F.col("revenue").desc()).limit(20))

product_id,unique_viewers,unique_purchasers,revenue,conversion_rate_pct
1005115,170989,8352,1.240483595E7,4.8845
1005105,114813,4794,1.023924868E7,4.1755
1004249,96989,5538,6729380.83,5.7099
1005135,62646,2163,5567806.64,3.4527
1004767,175572,14410,5430222.72,8.2075
1002544,89025,6781,4854785.55,7.617
1004856,197840,19228,3798168.71,9.719
1002524,51704,4132,3538299.12,7.9916
1003317,56575,2179,3051294.26,3.8515
1004870,84318,7331,3027098.05,8.6945


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecommerce;

USE CATALOG ecommerce;

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
USE CATALOG ecommerce;

CREATE OR REPLACE TABLE bronze.events AS
SELECT * FROM workspace.ecommerce_bronze.events;

CREATE OR REPLACE TABLE silver.events AS
SELECT * FROM workspace.ecommerce_silver.events;

CREATE OR REPLACE TABLE gold.product_perf AS
SELECT * FROM workspace.ecommerce_gold.product_perf;

num_affected_rows,num_inserted_rows


In [0]:
%sql
GRANT USE CATALOG ON CATALOG ecommerce TO `analysts@company.com`;
GRANT USE SCHEMA  ON SCHEMA ecommerce.gold TO `analysts@company.com`;
GRANT SELECT      ON TABLE  ecommerce.gold.product_perf TO `analysts@company.com`;

GRANT USE CATALOG ON CATALOG ecommerce TO `engineers@company.com`;
GRANT USE SCHEMA  ON SCHEMA ecommerce.silver TO `engineers@company.com`;
GRANT ALL PRIVILEGES ON SCHEMA ecommerce.silver TO `engineers@company.com`;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7333507100953673>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'GRANT USE CATALOG ON CATALOG ecommerce TO `analysts@company.com`;\nGRANT USE SCHEMA  ON SCHEMA ecommerce.gold TO `analysts@company.com`;\nGRANT SELECT      ON TABLE  ecommerce.gold.product_perf TO `analysts@company.com`;\n\nGRANT USE CATALOG ON CATALOG ecommerce TO `engineers@company.com`;\nGRANT USE SCHEMA  ON SCHEMA ecommerce.silver TO `engineers@company.com`;\nGRANT ALL PRIVILEGES ON SCHEMA ecommerce.silver TO `engineers@company.com`;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the outpu

In [0]:
%sql
USE CATALOG ecommerce;
USE SCHEMA gold;

CREATE OR REPLACE VIEW top_products AS
SELECT
  product_id,
  revenue,
  conversion_rate_pct
FROM gold.product_perf
WHERE unique_purchasers > 10
ORDER BY revenue DESC
LIMIT 100;

GRANT SELECT ON VIEW ecommerce.gold.top_products TO `analysts@company.com`;